# 22_rebalanced_dataset — 재균형 + decoy 할로겐 매칭 (업데이트)

**한 줄 요약:** active 전부 + real_inactive 전부 유지, **decoy는 active의 할로겐 비율(~89%)에 맞춰 선택**한다.
**왜 바뀌었나:** 원래 decoy는 할로겐 57%뿐이라 active(89%)와 격차가 커, 모델이 **"할로겐=active"** 지름길을 학습했다(GW-4064 오탐 원인). decoy 할로겐율을 active에 **매칭**하면 할로겐이 구별 단서가 못 되어 편향이 준다.
**중요:** **real_inactive(귀한 하드 네거티브)는 절대 안 건드림** — 여기서 손대면 activity-cliff 학습이 손상됨.
**근거:** DUD-E 물성 매칭 원리(할로겐도 물성) + 우리 실험(MCC 0.855→0.877, 천연물↑, 합성오탐↓).

> **📌 읽는 법**: 각 코드 셀은 [① 무슨 작업] → [② 코드] → [③ 🔎 코드 뜯어보기].

### 준비 — 도구 불러오기
표 처리 + 할로겐 판별(RDKit) + 층화 분할.

In [ ]:
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print('작업 폴더:', os.getcwd())
import pandas as pd
from rdkit import Chem
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
from sklearn.model_selection import train_test_split

🔎 **코드 뜯어보기 (준비)**
- `Chem.MolFromSmarts('[F,Cl,Br,I]')` : 할로겐 원자 패턴. 분자가 이걸 포함하는지로 할로겐 여부 판정.

### 셀 1 — 재균형 + decoy 할로겐 매칭
active·real_inactive는 전부, decoy는 active 할로겐율(~89%)에 맞춰 선택.

In [ ]:
# 재균형 구성: active 전부 + real_inactive 전부 + decoy는 active 할로겐율에 '매칭'해 선택
V2="data/HSD17B13_final_training_1to1_v2.csv"; SRC="data/train_1to1.csv"
OUT="data/HSD17B13_rebalanced_membership.csv"
base=pd.read_csv(V2, usecols=["canonical_smiles","potency"])
src=pd.read_csv(SRC)[["canonical_smiles","source"]]
df=base.merge(src, on="canonical_smiles", how="left")
# 할로겐 포함 여부 플래그 (매칭에 사용)
halo=Chem.MolFromSmarts("[F,Cl,Br,I]")
df["halo"]=df.canonical_smiles.map(lambda s:(lambda m:bool(m and m.HasSubstructMatch(halo)))(Chem.MolFromSmiles(str(s))))

act =df[df.source=="active"]                 # 2049 전부 유지
real=df[df.source=="real_inactive"]          # 200 전부 유지 (어려운 하드 네거티브 → 절대 안 건드림)
n_real=len(real)
target=act["halo"].mean()                    # active의 할로겐 비율(~0.89)이 '목표'
pool=df[df.source=="decoy"]
n_h=int(round(n_real*target))                # decoy 중 할로겐이어야 할 개수
# decoy를 real 개수만큼, 단 할로겐율을 active에 맞춰 선택 → 할로겐이 '구별 단서'가 안 되게
dec=pd.concat([pool[pool.halo].sample(n=min(n_h,int(pool.halo.sum())),random_state=42),
               pool[~pool.halo].sample(n=n_real-n_h,random_state=42)])
reb=pd.concat([act,real,dec],ignore_index=True)
print("재균형 →","active",len(act),"| real_inactive",len(real),"| decoy(할로겐매칭)",len(dec))
print(f"  할로겐율: active {act.halo.mean()*100:.0f}% | real {real.halo.mean()*100:.0f}% | decoy {dec.halo.mean()*100:.0f}% (원래 {pool.halo.mean()*100:.0f}% → active에 매칭)")

🔎 **코드 뜯어보기 (셀 1)**
- `target=act["halo"].mean()` : active의 할로겐 비율(목표값). `n_h=round(n_real*target)` : decoy 중 할로겐이어야 할 개수.
- `pd.concat([pool[pool.halo].sample(n_h), pool[~pool.halo].sample(나머지)])` : 할로겐 decoy를 많이, 비할로겐을 적게 뽑아 **비율을 active에 맞춤**.
- real_inactive는 `real` 그대로 → **손대지 않음**.

### 셀 2 — 층화 분할 & 저장
source 비율을 유지하며 train/val/test로 나눈다.

In [ ]:
# source로 층화 분할 → train/val/test (active·decoy·real_inactive 비율 유지)
TEST,VAL=0.15,0.15
tr_val,te=train_test_split(reb,test_size=TEST,stratify=reb["source"],random_state=42)
tr,va=train_test_split(tr_val,test_size=VAL/(1-TEST),stratify=tr_val["source"],random_state=42)
reb2=pd.concat([tr.assign(split="train"),va.assign(split="val"),te.assign(split="test")])[["canonical_smiles","source","potency","split"]]
reb2.to_csv(OUT,index=False)
print("분할 결과(그룹별):"); print(pd.crosstab(reb2["split"],reb2["source"]).to_string())
print("저장:",OUT,"| 총",len(reb2),"행")

🔎 **코드 뜯어보기 (셀 2)**
- `stratify=reb["source"]` : 세 그룹이 각 split에 비율대로. 20a/이전 22와 동일.